# 2 · Ejercicio 1: Curry-Howard (25 min)

**Idea:** demostrar una fórmula = escribir una función de Scala con ese tipo. Si compila sin `???`, está demostrada.

En los comentarios de cada ejemplo se apunta, en cada paso, la **meta** (el tipo que falta
por construir) y los **recursos** (las variables que ya tenemos). Es la forma de pensar que
hay que repetir en el examen.

### Preámbulo
Ejecuta esta celda antes que las demás: son las mismas definiciones que trae el examen, comentadas.

In [1]:
// ─────────────────────────────────────────────────────────────────────────────
// PREÁMBULO — las mismas definiciones que te dan en el examen.
// Ejecuta esta celda antes que ninguna otra del notebook.
// ─────────────────────────────────────────────────────────────────────────────

// Librería de tests. `run(TestX(f))` ejecuta los tests de un apartado.
import $ivy.`org.scalatest::scalatest:3.2.16`
import org.scalatest.{Filter => _, _}, flatspec._, matchers._

// "no P" se representa como una función que, si le das una P, produce Nothing
// (un tipo SIN valores = una contradicción). Si existe, es que P es imposible.
type Not[P] = P => Nothing
// "P si y solo si Q" = un par de funciones: de P a Q y de Q a P.
type <=>[P, Q] = (P => Q, Q => P)

// Tercio excluso: para cualquier P, o tengo una P o tengo una prueba de que no hay P.
// Se usa escribiendo  lem[X]  y haciendo match sobre el Either que devuelve.
trait ExcludedMiddle:
    def apply[P]: Either[P, Not[P]]

// Doble negación: si "no es verdad que no P", entonces P.
// Se usa escribiendo  dn[X](nx => ...)  cuando la meta es X.
trait DoubleNegation:
    def apply[P]: Not[Not[P]] => P

// Plantilla de los isomorfismos: tú implementas from y to; las leyes las usan los tests.
trait Isomorphic[A, B]:
    def from(a: A): B
    def to(b: B): A
    def equalA(a1: A, a2: A): Boolean = a1 == a2
    def equalB(b1: B, b2: B): Boolean = b1 == b2
    def law1(a: A): Boolean = equalA(to(from(a)), a)   // ir y volver deja igual una A
    def law2(b: B): Boolean = equalB(from(to(b)), b)   // ir y volver deja igual una B

// Árbol binario: o vacío, o un nodo con hijo izquierdo, raíz e hijo derecho.
enum Tree[A]:
    case Empty()
    case Node(left: Tree[A], root: A, right: Tree[A])

// Atajos para escribir árboles en los tests. El orden de los argumentos imita el dibujo:
// left(hijo, raiz) pone el hijo a la izquierda; right(raiz, hijo) a la derecha.
object Tree:
    def void[A]: Tree[A] = Empty()
    def leaf[A](a: A): Node[A] = Node(Empty(), a, Empty())
    def right[A](a: A, tree: Tree[A]): Node[A] = Node(Empty(), a, tree)
    def left[A](tree: Tree[A], a: A): Node[A] = Node(tree, a, Empty())
    def node[A](left: Tree[A], a: A, right: Tree[A]): Node[A] = Node(left, a, right)

import Tree._   // para escribir leaf(1) en vez de Tree.leaf(1)

Downloaded https://repo1.maven.org/maven2/org/scalatest/scalatest_3/3.2.16/scalatest_3-3.2.16.pom
Downloaded https://repo1.maven.org/maven2/org/scalatest/scalatest-refspec_3/3.2.16/scalatest-refspec_3-3.2.16.pom
Downloaded https://repo1.maven.org/maven2/org/scalatest/scalatest-flatspec_3/3.2.16/scalatest-flatspec_3-3.2.16.pom
Downloaded https://repo1.maven.org/maven2/org/scalatest/scalatest-featurespec_3/3.2.16/scalatest-featurespec_3-3.2.16.pom
Downloaded https://repo1.maven.org/maven2/org/scalatest/scalatest-core_3/3.2.16/scalatest-core_3-3.2.16.pom
Downloaded https://repo1.maven.org/maven2/org/scalatest/scalatest-funsuite_3/3.2.16/scalatest-funsuite_3-3.2.16.pom
Downloaded https://repo1.maven.org/maven2/org/scalatest/scalatest-wordspec_3/3.2.16/scalatest-wordspec_3-3.2.16.pom
Downloaded https://repo1.maven.org/maven2/org/scalatest/scalatest-mustmatchers_3/3.2.16/scalatest-mustmatchers_3-3.2.16.pom
Downloaded https://repo1.maven.org/maven2/org/scalatest/scalatest-shouldmatchers_3/3.2

import $ivy.$                                

import org.scalatest.{Filter => _, _}, flatspec._, matchers._

// "no P" se representa como una función que, si le das una P, produce Nothing
// (un tipo SIN valores = una contradicción). Si existe, es que P es imposible.

defined type Not
defined type <=>
defined trait ExcludedMiddle
defined trait DoubleNegation
defined trait Isomorphic
defined class Tree
defined object Tree
import Tree._   // para escribir leaf(1) en vez de Tree.leaf(1)


## Paso 1: traducir

| Lógica | Scala |
|---|---|
| p → q | `P => Q` |
| p ∧ q | `(P, Q)` |
| p ∨ q | `Either[P, Q]` |
| ¬p | `Not[P]` (= `P => Nothing`) |
| ⊥ | `Nothing` |

**Precedencia:** ¬ antes que ∧, antes que ∨, antes que →. La → agrupa a la **derecha**: `a → b → c` = `a → (b → c)`.

- `p ∧ q → r` es `((P, Q)) => R` (¡doble paréntesis: recibe **un** par!).
- Un argumento `{premisa1, premisa2} ⊢ conclusión` es una función con parámetros `(p1: ..., p2: ...)` que devuelve la conclusión.

## Paso 2: construir con "meta" y "recursos"

La **meta** es el tipo que te falta construir; los **recursos**, las variables que tienes.

| Si la meta es… | escribe |
|---|---|
| `P => Q` o `Not[P]` | `p => ...` (ganas `p`) |
| `(P, Q)` | `(..., ...)` |
| `Either[P, Q]` | `Left(...)` o `Right(...)` |
| `Nothing` | aplica una negación que tengas: `np(p)` |

| Si tienes… | puedes |
|---|---|
| `f: P => Q` y `p: P` | `f(p): Q` |
| `t: (P, Q)` | `t._1`, `t._2` o `(p, q) =>` |
| `e: Either[P, Q]` | `e match { case Left(p) => ...; case Right(q) => ... }` |
| algo de tipo `Nothing` | ¡úsalo como **cualquier** tipo! |

## Ejemplo 1 · ⊢ (p → q) → (q → r) → p → r

In [2]:
// Traducción: cada → es un =>, y como asocia a la derecha el tipo es
//   (P => Q) => ((Q => R) => (P => R))
def proof[P, Q, R]: (P => Q) => (Q => R) => P => R =
    // La meta es una función (tiene flechas) → la construyo con lambdas.
    // Cada lambda "consume" una flecha y me da un recurso nuevo:
    //   f: P => Q      g: Q => R      p: P
    // Tras las tres lambdas, la meta que queda es R.
    f => g => p =>
        // ¿Qué recurso devuelve R?  g, si le doy una Q.
        // ¿Cómo consigo una Q?      f, si le doy una P.  ¡Y tengo p!
        // Se lee de dentro a fuera: f(p) es una Q, y g de esa Q es una R.
        g(f(p))

defined function proof

## Ejemplo 2 · ⊢ ¬p ∨ ¬q → ¬(p ∧ q)

Tenemos un `Either` → `match`. La meta de cada rama es `Not[(P, Q)]` → lambda que recibe el par.

In [ ]:
// Traducción: (¬p ∨ ¬q) → ¬(p ∧ q)  =  Either[Not[P], Not[Q]] => Not[(P, Q)]
def proof[P, Q]: Either[Not[P], Not[Q]] => Not[(P, Q)] =
    // El argumento es un Either → para usarlo hay que ver cuál de los dos lados es.
    // Escribimos directamente los case (equivale a  e => e match ...).
    // En CADA rama la meta es la misma: Not[(P, Q)], que es  ((P, Q)) => Nothing.

    case Left(np) =>        // recurso np: Not[P]   (sé que p es imposible)
        // Meta Not[(P,Q)] = función que recibe un par → lambda que recibe el par (p, q).
        // Ahora la meta es Nothing. np(p) es exactamente un Nothing.
        // La q no la necesito, por eso la llamo _.
        (p, _) => np(p)

    case Right(nq) =>       // recurso nq: Not[Q]
        // Mismo razonamiento, pero ahora la contradicción viene de q.
        (_, q) => nq(q)

## Ejemplo 3 · {p → q ∨ r, ¬r} ⊢ p → q (con premisas)

In [3]:
// Las premisas (lo que hay a la izquierda de ⊢) se convierten en PARÁMETROS:
//   p1: P => Either[Q, R]     (p → q ∨ r)
//   p2: Not[R]                (¬r)
// y la conclusión (p → q) es el tipo de lo que se devuelve.
def proof[P, Q, R](p1: P => Either[Q, R], p2: Not[R]): P => Q =
    p =>                         // meta P => Q → lambda. Recurso nuevo: p. Meta: Q
        p1(p) match              // con p, p1 me da un Either[Q, R] → miro cuál es
            case Left(q)  => q   // si es una Q, ya tengo la meta
            case Right(r) =>     // si es una R... ¡pero p2 dice que R es imposible!
                p2(r)            // p2(r) es un Nothing, y Nothing vale como cualquier
                                 // tipo (también como Q). Esta rama "nunca ocurre".

defined function proof

In [ ]:
## Apartado b): lógica clásica

Te dejan usar una ayuda **sobre una proposición concreta** (la que diga el enunciado):

- **Tercio excluso** (`lem: ExcludedMiddle`): `lem[X]` te da `Either[X, Not[X]]` → haz `match`.
- **Doble negación** (`dn: DoubleNegation`): si la meta es `X`, escribe `dn[X](nx => ...)`; dentro la meta es `Nothing` y tienes `nx: Not[X]`.

In [ ]:
## Ejemplo 4 · ⊢ (a → b) → ¬a ∨ b, con tercio excluso en a

In [ ]:
// Sin ayuda no se puede: para devolver Either[Not[A], B] hay que ELEGIR un lado
// (Left o Right), y con solo  f: A => B  no sabemos cuál. El tercio excluso nos dice
// en qué caso estamos.
def proof[A, B](lem: ExcludedMiddle): (A => B) => Either[Not[A], B] =
    f =>                        // recurso f: A => B.  Meta: Either[Not[A], B]
        lem[A] match            // lem[A] : Either[A, Not[A]]  (usamos A porque lo dice el enunciado)
            case Left(a)   =>   // caso "hay una a": con f la convierto en b
                Right(f(a))     //   → elijo el lado derecho (b)
            case Right(na) =>   // caso "a es imposible": eso es justo el lado izquierdo
                Left(na)

## Ejemplo 5 · ⊢ ((p → q) → p) → p, con doble negación en p

In [ ]:
def proof[P, Q](dn: DoubleNegation): ((P => Q) => P) => P =
    f =>                        // recurso f: (P => Q) => P.   Meta: P
        // La meta es P y nos dejan usar doble negación en P → dn[P](np => ...).
        // dn[P] pide un Not[Not[P]], es decir, una función  Not[P] => Nothing.
        // Dentro: recurso nuevo np: Not[P], y la meta pasa a ser Nothing.
        dn[P](np =>
            // Para llegar a Nothing aplico np a una P. ¿De dónde saco una P?  De f,
            // si le doy una función P => Q.  np es P => Nothing, y como Nothing vale
            // como cualquier tipo, np también sirve como P => Q.
            //   f(np)      : P
            //   np(f(np))  : Nothing  ✓
            np(f(np))
        )

## Mini-ejercicios (5 min)

Rellena los `???` y comprueba que compila.

**1.** ⊢ p ∧ q → q ∧ p

In [ ]:
def ej1[P, Q]: ((P, Q)) => (Q, P) =
    (p,q) => (q,p)

<details>
<summary><b>Ver solución</b></summary>

```scala
def ej1[P, Q]: ((P, Q)) => (Q, P) =
    // Meta: función que recibe UN par → lambda que lo desempaqueta en p y q.
    // Luego construyo el par de salida con los componentes cambiados de orden.
    (p, q) => (q, p)
```

</details>

**2.** ⊢ (p → q) → ¬q → ¬p

In [ ]:
def ej2[P, Q]: (P => Q) => Not[Q] => Not[P] =
    f => 
        nq =>
            np(f(p))

<details>
<summary><b>Ver solución</b></summary>

```scala
def ej2[P, Q]: (P => Q) => Not[Q] => Not[P] =
    // Tres flechas en la meta (Not[P] también es una flecha: P => Nothing) → tres lambdas.
    // Recursos: f: P => Q, nq: Not[Q], p: P.   Meta: Nothing.
    // Nothing sale de aplicar nq a una Q; la Q sale de f(p).
    f => nq => p => nq(f(p))
```

</details>

**3.** ⊢ ¬¬p → p, con tercio excluso en p

In [4]:
def ej3[P](lem: ExcludedMiddle): Not[Not[P]] => P =
    nnp => 
        lem[P] match
            case Left(p) => p
            case Right(np) => nnp(np)

defined function ej3

<details>
<summary><b>Ver solución</b></summary>

```scala
def ej3[P](lem: ExcludedMiddle): Not[Not[P]] => P =
    nnp =>                      // recurso nnp: Not[Not[P]] = Not[P] => Nothing.  Meta: P
        lem[P] match            // ¿hay P o no la hay?
            case Left(p)   => p             // la hay: es la meta
            case Right(np) => nnp(np)       // no la hay: nnp(np) es Nothing → vale como P
```

</details>

**Siguiente:** `03-ejercicio2-isomorfismos.ipynb`